# Merge de los datos de transaccion

In [4]:
import pandas as pd
import numpy as np
import json
import warnings
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap

In [5]:
df = pd.read_csv("data/dataset_transacciones/hey_clientes_perfil_completo.csv")
df

,user_id,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,...,monto_total_txn,monto_promedio_txn,cashback_total,txn_internacionales,categorias_unicas,canales_usados,tipo_producto_principal,categoria_mcc_principal,canal_principal_real,tipo_operacion_principal
0,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,...,119570.59,2135.189107,122.33,2,7,"app_android, app_huawei, app_ios, codi, oxxo, ...",cuenta_debito,servicios_digitales,app_android,compra
1,USR-00002,18,M,Jalisco,Puerto Vallarta,Preparatoria,Estudiante,19000,1410,True,...,212722.67,2874.630676,147.08,1,7,"app_android, app_huawei, app_ios, cajero_banre...",cuenta_debito,servicios_digitales,app_android,compra
2,USR-00003,23,H,Chihuahua,Cuauhtémoc,Licenciatura,Estudiante,14000,1174,True,...,190801.09,2415.203671,165.43,4,8,"app_android, app_huawei, app_ios, cajero_banre...",cuenta_debito,servicios_digitales,app_android,compra
3,USR-00004,32,SE,Nuevo León,Guadalupe,Posgrado,Empleado,61000,1168,False,...,580113.74,8789.602121,0.00,2,10,"app_android, app_huawei, app_ios, cajero_banre...",cuenta_debito,supermercado,app_android,compra
4,USR-00005,26,M,Ciudad de México,CDMX - Cuauhtémoc,Preparatoria,Empresario,27000,816,True,...,190277.30,2045.992473,162.99,1,8,"app_android, app_huawei, app_ios, cajero_banre...",cuenta_debito,servicios_digitales,app_ios,compra
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15020,USR-15021,56,M,Nuevo León,Monterrey,Secundaria,Empleado,40000,1145,True,...,565510.87,9584.930000,428.84,5,10,"app_android, app_huawei, app_ios, cajero_exter...",cuenta_debito,supermercado,app_android,compra
15021,USR-15022,34,M,Ciudad de México,CDMX - Miguel Hidalgo,Licenciatura,Empleado,13500,1652,False,...,306412.43,5023.154590,0.00,5,9,"app_android, app_huawei, app_ios, cajero_banre...",cuenta_debito,gobierno,pos_fisico,compra
15022,USR-15023,26,M,Ciudad de México,CDMX - Cuauhtémoc,Preparatoria,Empleado,19500,671,True,...,230110.76,4183.832000,79.30,1,8,"app_android, app_huawei, app_ios, cajero_banre...",cuenta_debito,servicios_digitales,app_ios,compra
15023,USR-15024,27,M,Ciudad de México,CDMX - Benito Juárez,Licenciatura,Empleado,4500,434,False,...,55102.40,4238.646154,0.00,1,4,"app_android, app_huawei, app_ios, cajero_banre...",cuenta_debito,gobierno,app_ios,compra


In [6]:
# Separar features por tipo
numericas = [
    'edad',
    'ingreso_mensual_mxn',
    'antiguedad_dias',
    'score_buro',
    'dias_desde_ultimo_login',
    'satisfaccion_1_10',
    'num_productos_activos',
    'limite_credito_total',
    'saldo_total',
    'utilizacion_promedio',
    'total_transacciones',
    'monto_total_txn',
    'monto_promedio_txn',
    'cashback_total',
    'txn_internacionales',
    'categorias_unicas',
]
 
binarias = [
    'es_hey_pro',
    'nomina_domiciliada',
    'recibe_remesas',
    'usa_hey_shop',
    'tiene_seguro',
    'patron_uso_atipico',
]
 
categoricas = [
    'sexo',
    'nivel_educativo',
    'ocupacion',
    'canal_apertura',
    'preferencia_canal',
    'idioma_preferido',
    'tipo_producto_principal',
    'categoria_mcc_principal',
    'canal_principal_real',
    'tipo_operacion_principal',
]

# Preprocesamiento
dm = df[numericas + binarias + categoricas].copy()

dm[numericas] = dm[numericas].fillna(dm[numericas].median())
dm[binarias]  = dm[binarias].fillna(False).astype(int)

for col in categoricas:
    dm[col] = dm[col].fillna(dm[col].mode()[0])
    le = LabelEncoder()
    dm[col] = le.fit_transform(dm[col].astype(str))

# Escalar variables numéricas
scaler = StandardScaler()
dm[numericas] = scaler.fit_transform(dm[numericas])
 
X = dm.values.astype(np.float32)

In [7]:
# =============================================================================
# PASO 4 — REDUCCIÓN DIMENSIONAL
# =============================================================================
# Objetivo: eliminar ruido y correlaciones antes de clusterizar.
# Con 32 features muchas son redundantes (ej. monto_total_txn ≈ txn * monto_prom).
#
# Opción A — PCA (usado aquí por limitaciones de RAM):
#   Rápido, determinista, funciona bien con variables numéricas ya escaladas.
#   Captura estructura lineal. 10 componentes = 80.1% de varianza explicada.
#
# Opción B — UMAP (recomendado en producción):
#   Preserva estructura no-lineal y clusters locales. Mejor para datos mixtos.
#   Requiere más memoria (~4-8 GB para 15k filas × 32 dims).
#
# Descomenta el bloque UMAP si tu entorno tiene suficiente RAM:
 
# --- UMAP (producción) ---
# reducer = umap.UMAP(
#     n_components=10,
#     n_neighbors=30,   # vecinos para construir el grafo; más = más global
#     min_dist=0.1,     # qué tan juntos pueden quedar puntos en el embedding
#     metric='euclidean',
#     random_state=42
# )
# embedding = reducer.fit_transform(X)
# --- PCA (este entorno) ---
pca = PCA(n_components=10, random_state=42)
embedding = pca.fit_transform(X)
var_explicada = pca.explained_variance_ratio_.sum() * 100

In [8]:
# =============================================================================
# PASO 5 — BÚSQUEDA DEL K ÓPTIMO (SILHOUETTE SCORE + MÉTODO DEL CODO)
# =============================================================================
# Silhouette score: mide qué tan bien separado está cada punto de su cluster
# vs. el cluster más cercano. Rango [-1, 1]. Más alto = mejor separación.
#
# Método del codo (inertia): suma de distancias cuadradas al centroide.
# Buscamos el punto donde la curva "dobla" y deja de bajar rápidamente.
#
# sample_size en silhouette_score: calcular con todos los puntos es O(n²).
# Una muestra de 5,000 es estadísticamente representativa para 15k registros.
 
print("[5] Evaluando k=2 a k=8:")
sil_scores = {}
inertias   = {}
 
for k in range(2, 9):
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embedding)
 
    sil = silhouette_score(embedding, labels, sample_size=5000, random_state=42)
    sil_scores[k] = round(float(sil), 4)
    inertias[k]   = round(float(km.inertia_), 2)
 
    print(f"    k={k}: silhouette={sil:.4f}  inertia={km.inertia_:,.0f}")
 
k_matematico = max(sil_scores, key=sil_scores.get)
print(f"\n    K matemáticamente óptimo: {k_matematico} "
      f"(silhouette={sil_scores[k_matematico]})")
 

[5] Evaluando k=2 a k=8:
    k=2: silhouette=0.3159  inertia=279,150
    k=3: silhouette=0.2984  inertia=244,501
    k=4: silhouette=0.2491  inertia=211,549
    k=5: silhouette=0.2414  inertia=188,649
    k=6: silhouette=0.2468  inertia=175,884
    k=7: silhouette=0.2264  inertia=165,676
    k=8: silhouette=0.1947  inertia=160,141

    K matemáticamente óptimo: 2 (silhouette=0.3159)


In [9]:
# =============================================================================
# PASO 6 — MODELO FINAL
# =============================================================================
# Usamos k=5 en lugar del k=2 matemáticamente óptimo.
# Razón: con k=2 los segmentos son "alto score" vs "bajo score", lo cual
# es verdad pero poco accionable. Con k=5 obtenemos arquetipos diferenciados
# en ingreso, comportamiento digital, antigüedad y productos activos.
# Esta es una decisión deliberada de utilidad de negocio sobre pureza estadística.
#
# Regla práctica: en perfilado de clientes bancarios, k entre 4 y 7 suele
# dar el mejor balance entre interpretabilidad y granularidad.
 
K_ELEGIDO = 5
 
km_final = KMeans(n_clusters=K_ELEGIDO, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(embedding)
 
print(f"\n[6] Modelo final K-Means con k={K_ELEGIDO}:")
print(df['cluster'].value_counts().sort_index().to_string())
 


[6] Modelo final K-Means con k=5:
cluster
0    3213
1    3422
2    2068
3    4789
4    1533


In [10]:
# =============================================================================
# PASO 7 — UMAP 2D PARA VISUALIZACIÓN
# =============================================================================
# El UMAP 10D para clustering requiere demasiada RAM con el dataset completo.
# Para visualización usamos una muestra de 3,000 registros con UMAP 2D,
# que sí es manejable y da una representación visual fiel de los clusters.
 
print("\n[7] Generando UMAP 2D para visualización (muestra 3,000 registros)...")
idx_sample = np.random.RandomState(42).choice(len(X), size=3000, replace=False)
X_sample   = X[idx_sample]
 
reducer_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
emb_2d     = reducer_2d.fit_transform(X_sample)
 
# Guardar coordenadas 2D en el dataframe (NaN para los no muestreados)
df['umap_x'] = np.nan
df['umap_y'] = np.nan
df.loc[idx_sample, 'umap_x'] = emb_2d[:, 0]
df.loc[idx_sample, 'umap_y'] = emb_2d[:, 1]
 
print("    UMAP 2D OK")
 


[7] Generando UMAP 2D para visualización (muestra 3,000 registros)...


c:\Users\ludio\Datathon-2026-Wicho\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


    UMAP 2D OK


In [ ]:
# =============================================================================
# PASO 9 — PERFIL DE CADA CLUSTER
# =============================================================================
# Calculamos estadísticas descriptivas por cluster para poder nombrar e
# interpretar cada segmento de negocio.
 
print("\n[9] Perfil de clusters:\n")
 
perfil = df.groupby('cluster').agg(
    n_clientes       = ('user_id',                'count'),
    edad_prom        = ('edad',                   'mean'),
    ingreso_prom     = ('ingreso_mensual_mxn',    'mean'),
    score_buro_prom  = ('score_buro',             'mean'),
    antiguedad_prom  = ('antiguedad_dias',        'mean'),
    txn_prom         = ('total_transacciones',    'mean'),
    monto_prom       = ('monto_promedio_txn',     'mean'),
    monto_total_prom = ('monto_total_txn',        'mean'),
    saldo_prom       = ('saldo_total',            'mean'),
    util_prom        = ('utilizacion_promedio',   'mean'),
    pct_hey_pro      = ('es_hey_pro',             'mean'),
    pct_nomina       = ('nomina_domiciliada',     'mean'),
    pct_seguro       = ('tiene_seguro',           'mean'),
    pct_remesas      = ('recibe_remesas',         'mean'),
    categoria_top    = ('categoria_mcc_principal', lambda x: x.mode()[0]),
    canal_top        = ('canal_principal_real',   lambda x: x.mode()[0]),
    producto_top     = ('tipo_producto_principal', lambda x: x.mode()[0]),
    operacion_top    = ('tipo_operacion_principal', lambda x: x.mode()[0]),
).round(2)
 
print(perfil.to_string())

In [ ]:
# =============================================================================
# PASO 10 — EXPORTAR MÉTRICAS
# =============================================================================
 
resultados = {
    'silhouette_scores' : {str(k): v for k, v in sil_scores.items()},
    'inertias'          : {str(k): v for k, v in inertias.items()},
    'k_matematico'      : k_matematico,
    'k_elegido'         : K_ELEGIDO,
    'pca_varianza_pct'  : round(var_explicada, 1),
    'perfil_clusters'   : perfil.reset_index().to_dict(orient='records'),
}
 
with open('hey_cluster_resultados.json', 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
 
print("\n[10] Métricas exportadas: hey_cluster_resultados.json")
print("\n=== PIPELINE COMPLETADO ===")